# 🍄 슈퍼마리오 DQN 학습
**런타임 → 런타임 유형 변경 → T4 GPU 선택 후 실행하세요**

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/supermario_dl_project'
os.makedirs(f'{DRIVE_PATH}/models', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/results', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/videos', exist_ok=True)
print('Drive 마운트 완료!')

## 2. 패키지 설치

In [ ]:
!apt-get install -y xvfb > /dev/null 2>&1

# nes-py 빌드 오류 방지 — setuptools 먼저 다운그레이드
!pip install setuptools==65.5.0 wheel -q

# nes-py 단독 설치 (빌드 순서 중요)
!pip install nes-py==8.2.1 -q

# 나머지 패키지 설치
!pip install gym==0.21.0 gym-super-mario-bros==7.4.0 -q
!pip install opencv-python-headless imageio imageio-ffmpeg pyvirtualdisplay tqdm -q

print('설치 완료!')

## 3. 가상 디스플레이 시작 (Colab 헤드리스 렌더링)

In [ ]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(400, 300))
display.start()
print('가상 디스플레이 시작!')

## 4. GitHub에서 프로젝트 클론

In [ ]:
GITHUB_REPO = 'https://github.com/HyunDove/supermario-dqn.git'

import os
if not os.path.exists('/content/supermario-dqn'):
    !git clone {GITHUB_REPO} /content/supermario-dqn
else:
    !cd /content/supermario-dqn && git pull

os.chdir('/content/supermario-dqn')
print('프로젝트 준비 완료!')

## 5. GPU 확인

In [ ]:
import torch
print(f'GPU 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 모델: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

## 6. 학습 실행 (영상 자동 기록 포함)

In [ ]:
import sys
sys.path.insert(0, '/content/supermario-dqn')

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.env.wrappers import make_env
from src.agent.dqn_agent import DQNAgent
from src.utils.recorder import record_episode

# ====== 설정 ======
EPISODES = 1000
SAVE_EVERY = 200          # 모델 체크포인트 저장 주기
RECORD_AT = [0, 1000, 5000, 7000]  # 영상 기록 에피소드
CHECKPOINT_PATH = None    # 이어서 학습: f'{DRIVE_PATH}/models/mario_ep1000.pth'
# ==================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env = make_env(world=1, stage=1)
n_actions = env.action_space.n
agent = DQNAgent(n_actions=n_actions, device=device)

start_episode = 0
rewards_history = []
best_reward = -float('inf')

if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
    agent.load(CHECKPOINT_PATH)
    start_episode = int(CHECKPOINT_PATH.split('ep')[-1].split('.')[0])
    print(f'에피소드 {start_episode}부터 재개')

# --- 0회 영상 기록 (학습 전 랜덤 에이전트) ---
if 0 in RECORD_AT and start_episode == 0:
    print('\n[EP 0] 학습 전 영상 기록 중...')
    reward, frames = record_episode(
        lambda: make_env(1, 1), agent,
        f'{DRIVE_PATH}/videos/mario_ep0000.mp4', epsilon=1.0
    )
    print(f'[EP 0] 저장 완료 | 보상: {reward:.1f} | 프레임: {frames}')

# --- 학습 루프 ---
for episode in tqdm(range(start_episode, EPISODES)):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.select_action(state)
        next_state, reward, done, info = env.step(action)
        agent.store(state, action, reward, next_state, done)
        agent.learn()
        state = next_state
        total_reward += reward

    rewards_history.append(total_reward)
    current_ep = episode + 1

    # 체크포인트 저장
    if current_ep % SAVE_EVERY == 0:
        path = f'{DRIVE_PATH}/models/mario_ep{current_ep}.pth'
        agent.save(path)
        avg = np.mean(rewards_history[-100:])
        print(f'\n[EP {current_ep}] 평균보상: {avg:.1f} | epsilon: {agent.epsilon:.3f}')

    # 최고 보상 모델 저장
    if total_reward > best_reward:
        best_reward = total_reward
        agent.save(f'{DRIVE_PATH}/models/mario_best.pth')

    # 영상 기록 (1000, 5000, 7000)
    if current_ep in RECORD_AT:
        saved_eps = agent.epsilon
        print(f'\n[EP {current_ep}] 영상 기록 중...')
        reward, frames = record_episode(
            lambda: make_env(1, 1), agent,
            f'{DRIVE_PATH}/videos/mario_ep{current_ep:04d}.mp4', epsilon=0.05
        )
        agent.epsilon = saved_eps  # 영상 기록 후 epsilon 복원
        print(f'[EP {current_ep}] 저장 완료 | 보상: {reward:.1f} | 프레임: {frames}')

env.close()
print('\n학습 완료!')

## 7. 학습 곡선 그래프 저장

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('슈퍼마리오 DQN 학습 결과', fontsize=14, fontweight='bold')

# --- 상단: 에피소드별 보상 ---
ax1 = axes[0]
ax1.plot(rewards_history, alpha=0.3, color='steelblue', label='에피소드 보상')
window = min(100, len(rewards_history))
avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(rewards_history)), avg, color='tomato', linewidth=2, label=f'{window}회 이동 평균')

# 영상 기록 지점 표시
record_points = [ep for ep in [0, 1000, 5000, 7000] if ep < len(rewards_history)]
for ep in record_points:
    ax1.axvline(x=ep, color='gold', linestyle='--', alpha=0.7, linewidth=1.2)
    ax1.text(ep, ax1.get_ylim()[1]*0.9, f'EP{ep}\n영상', fontsize=7, ha='center', color='goldenrod')

ax1.set_xlabel('에피소드')
ax1.set_ylabel('총 보상')
ax1.set_title('에피소드별 보상 변화')
ax1.legend()
ax1.grid(alpha=0.3)

# --- 하단: 100회 이동 평균만 ---
ax2 = axes[1]
ax2.plot(range(window-1, len(rewards_history)), avg, color='tomato', linewidth=2)
ax2.fill_between(range(window-1, len(rewards_history)), avg, alpha=0.2, color='tomato')
ax2.set_xlabel('에피소드')
ax2.set_ylabel('평균 보상 (100회)')
ax2.set_title('학습 안정화 추세 (100회 이동 평균)')
ax2.grid(alpha=0.3)

plt.tight_layout()
save_path = f'{DRIVE_PATH}/results/training_curve.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'그래프 저장: {save_path}')

## 8. 저장된 파일 목록 확인

In [ ]:
import os

print('=== 저장된 모델 ===')
for f in sorted(os.listdir(f'{DRIVE_PATH}/models')):
    size = os.path.getsize(f'{DRIVE_PATH}/models/{f}') / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

print('\n=== 기록된 영상 ===')
for f in sorted(os.listdir(f'{DRIVE_PATH}/videos')):
    size = os.path.getsize(f'{DRIVE_PATH}/videos/{f}') / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

print('\n=== 결과 그래프 ===')
for f in sorted(os.listdir(f'{DRIVE_PATH}/results')):
    print(f'  {f}')

## 9. 세션 끊긴 후 이어서 학습하기

In [ ]:
# 1~5번 셀 재실행 후, 6번 셀에서 CHECKPOINT_PATH 수정
# 예시:
# CHECKPOINT_PATH = f'{DRIVE_PATH}/models/mario_ep1000.pth'

import os
print('마지막 저장 모델:')
models = sorted(os.listdir(f'{DRIVE_PATH}/models'))
for m in models[-3:]:
    print(f'  {DRIVE_PATH}/models/{m}')